# 02 · 数据准备与收益矩阵

**目标**：拉取 `01` 阶段产出的候选 ETF 历史收盘价，计算日收益、月收益、相关系数矩阵，存 parquet 供后续 notebook 使用。

**输入**：`etf_portfolio/outputs/universe.csv`

**输出**：
- `prices_daily.parquet`（收盘价宽表）
- `returns_daily.parquet`（日收益）
- `returns_monthly.parquet`（月收益）
- `corr_matrix.parquet`（相关系数矩阵）

In [ ]:
# ============================================================
# cell 0: imports + 全局参数
# ============================================================
from jqdata import *            # 聚宽 magic
import sys, os
from pathlib import Path
from datetime import datetime, date

import pandas as pd
import numpy as np

PROJ = Path('/Users/huhao/src/codesnip/python/ai/028-jukuan').resolve()
sys.path.insert(0, str(PROJ))

from etf_portfolio.data_loader import (
    fetch_prices, to_returns, to_monthly_returns,
    save_parquet, load_parquet,
)

AS_OF         = date(2026, 9, 8)
LOOKBACK_DAYS = 3 * 365 + 30      # 留 1 月 buffer
OUTPUT_DIR    = PROJ / 'etf_portfolio' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
# ============================================================
# cell 1: 加载候选池
# ============================================================
uni = pd.read_csv(OUTPUT_DIR / 'universe.csv', dtype={'code': str, 'list_date': str})
codes = uni['code'].tolist()
print(f'候选池 {len(codes)} 只:')
uni[['code', 'name', 'category']]

In [ ]:
# ============================================================
# cell 2: 拉历史收盘价（前复权）
# ============================================================
prices = fetch_prices(
    codes,
    end_date=AS_OF,
    lookback_days=LOOKBACK_DAYS,
)
print(f'价格矩阵 shape = {prices.shape}')
print(f'日期范围: {prices.index.min().date()} ~ {prices.index.max().date()}')
prices.head()

In [ ]:
# ============================================================
# cell 3: 检查缺失值 + 截取共同有效日期
# ============================================================
nan_ratio = prices.isna().mean()
print('缺失率 > 5% 的 ETF:')
print(nan_ratio[nan_ratio > 0.05])

# 截取每只 ETF 上市后的数据，避免早期 NaN
def trim_to_listing(df, uni_df):
    out = df.copy()
    for _, row in uni_df.iterrows():
        if row['code'] in out.columns:
            listing = pd.Timestamp(row['list_date'])
            out.loc[out.index < listing, row['code']] = np.nan
    return out

prices = trim_to_listing(prices, uni)
valid_counts = prices.notna().sum()
print(f'\n有效观测日最少: {valid_counts.min()} ({valid_counts.idxmin()})')
print(f'有效观测日最多: {valid_counts.max()} ({valid_counts.idxmax()})')

In [ ]:
# ============================================================
# cell 4: 收益转换
# ============================================================
ret_daily = to_returns(prices, log=False)
ret_monthly = to_monthly_returns(ret_daily)
print(f'日收益 shape = {ret_daily.shape}')
print(f'月收益 shape = {ret_monthly.shape}')
ret_daily.describe().T[['mean', 'std', 'min', 'max']]

In [ ]:
# ============================================================
# cell 5: 相关系数矩阵
# ============================================================
corr = ret_daily.corr()
print('相关系数矩阵（前 5 行 × 5 列）：')
print(corr.iloc[:5, :5].round(3))

In [ ]:
# ============================================================
# cell 6: 持久化
# ============================================================
save_parquet(prices,       OUTPUT_DIR / 'prices_daily.parquet')
save_parquet(ret_daily,    OUTPUT_DIR / 'returns_daily.parquet')
save_parquet(ret_monthly,  OUTPUT_DIR / 'returns_monthly.parquet')
save_parquet(corr,         OUTPUT_DIR / 'corr_matrix.parquet')
print('已写入 4 个 parquet：')
for p in ['prices_daily', 'returns_daily', 'returns_monthly', 'corr_matrix']:
    print(f'  {p}.parquet  ({ (OUTPUT_DIR / (p+chr(46)+"parquet")).stat().st_size / 1024:.1f} KB)')

In [ ]:
# ============================================================
# cell 7: 标的池日收益曲线对比图
# ============================================================
import matplotlib.pyplot as plt

# 归一化净值（每只 ETF 以起点为 1）
nav = (1 + ret_daily).cumprod()
nav = nav / nav.iloc[0]

# 用 name 而非 code 作图例
code2name = dict(zip(uni['code'], uni['name']))
nav_named = nav.rename(columns=code2name)

fig, ax = plt.subplots(figsize=(12, 7))
for col in nav_named.columns:
    ax.plot(nav_named.index, nav_named[col], label=col, linewidth=1.0, alpha=0.7)
ax.set_title('候选 ETF 累计净值曲线（起点归一）')
ax.set_xlabel('日期')
ax.set_ylabel('累计净值')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8, ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'candidates_nav.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# cell 8: 相关矩阵热图（带 name）
# ============================================================
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
labels = [code2name.get(c, c) for c in corr.columns]
ax.set_xticks(range(len(corr.columns)))
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_yticks(range(len(corr.index)))
ax.set_yticklabels(labels, fontsize=8)
plt.colorbar(im, ax=ax, label='相关系数')
ax.set_title('日收益相关系数矩阵')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'corr_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## 中间结论

- 已完成候选池历史行情拉取（前复权，覆盖 ~3 年）
- 已生成日 / 月收益矩阵、相关系数矩阵，全部落盘为 parquet
- 候选池日均有效观测天数应满足 WFA 60 个月训练窗需要
- 进入 `03_优化与回测框架.ipynb` 时直接读 parquet 即可